# Phase 7. Snowballing

**Md Forhadul Islam**. Review of benchmarks for coding agents and agentic software engineering (2022 to 2026).

Forward snowballing collects the papers that **cite** a seed (OpenAlex filter `cites:`); backward snowballing collects the
papers **cited by** the papers kept at stage 1 (filter `cited_by:`, 40 IDs per call). Round 2 repeats both from the papers
newly included in round 1, and each later round from the papers newly included in the round before it, until a round finds no
new included paper. The records of 9 and 10 November are in `data/` and were numbered in Phase 3. In an online run this notebook
repeats both directions today and compares; the saved run was offline, so sections 7.1 to 7.3 are skipped and only the saved counts
of earlier live runs, rounds 3 to 5 and the yield figures are shown and checked.

In [1]:
from setup import *
assert Path.cwd() == ROOT, "the notebook must work inside the Report folder"
print("Report folder:", ROOT)

Report folder: <Report folder>


In [2]:
import urllib.request, urllib.parse
LIVE = ROOT / 'snowballing' / f'live_rerun_{datetime.date.today().isoformat()}'; os.makedirs(LIVE, exist_ok=True)
def get(url):
    for i in range(4):
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'review-reproduction'}), timeout=90) as r: return json.loads(r.read())
        except Exception as e: print('retry', type(e).__name__); time.sleep(5 * (i + 1))
    raise RuntimeError('OpenAlex not reachable')
def earlier(name):
    # OpenAlex limits the calls per day; when the limit is reached the result of the latest earlier run is shown instead
    c = sorted(glob.glob(str(ROOT / 'snowballing' / 'live_rerun_*' / name)))
    if not c: raise RuntimeError('OpenAlex not reachable and no earlier run saved')
    print('OpenAlex daily limit reached: using the run saved in', Path(c[-1]).parent.name); return set(json.load(open(c[-1])))
def ids_for(filter_text, max_pages=15):
    page, out = 1, set()
    while True:
        j = get(f'https://api.openalex.org/works?filter={filter_text}&per-page=200&select=id&page={page}')
        out |= {w['id'] for w in j['results']}
        if page * 200 >= j['meta']['count'] or page >= max_pages: return out
        page += 1; time.sleep(0.2)
FW = J('data/openalex_forward_snowball_round1_2026-11-09.json'); SA = J('data/openalex_forward_sweagent_and_seeds_2026-11-09.json')
BW = J('data/openalex_backward_snowball_round1_2026-11-09.json'); R2 = J('data/openalex_snowball_round2_2026-11-10.json')
seeds = [(l['seed'], l['seed_id']) for l in FW['log']] + [('SWE-agent', 'W7103754783')]

## 7.1 Forward, today

In [3]:
if OFFLINE:
    print("offline test run: this live internet step is skipped")
else:
    fw_today = set()
    try:
        for name, sid in seeds:
            s = ids_for(f'cites:{sid},from_publication_date:2022-11-01', max_pages=50); fw_today |= s; print(f'{len(s):5}  {name[:70]}')
    except RuntimeError: fw_today = earlier('forward_today_ids.json')
    old = {r['id'] for r in FW['records']} | {r['id'] for r in SA['records']}
    print(f'forward today {len(fw_today)}, 9 November {len(old)}, found both times {len(fw_today & old)}')
    json.dump(sorted(fw_today), open(LIVE / 'forward_today_ids.json', 'w'))

offline test run: this live internet step is skipped


## 7.2 Backward, today

The start set is the papers marked Include or Maybe at stage 1 among the database search records (R) and the round 1 forward records (F, S). Google Scholar records have no OpenAlex ID and the seed records are the forward starting points, so neither is in the start set.

In [4]:
if OFFLINE:
    print("offline test run: this live internet step is skipped")
else:
    import openpyxl
    M = J('screening/stage1_title_abstract/decisions/stage1_all_decisions_merged.json')
    oid = {}
    for rel, key in [('snowballing/round_1_forward/forward_new_records_with_ids.json', 'fid'), ('snowballing/round_1_forward/sweagent_new_records_with_ids.json', 'sid')]:
        for r in J(rel): oid[r[key]] = r['id']
    ws = openpyxl.load_workbook(ROOT / 'tracker/master_tracker/Master_Tracker_LATEST.xlsx', read_only=True)['All_Records']
    it = ws.iter_rows(values_only=True); h = list(next(it))
    for r in it: oid.setdefault(r[h.index('ID')], r[h.index('OpenAlex ID')])
    start = [oid[d['id']].replace('https://openalex.org/', '') for d in M if d['decision'] in ('Include', 'Maybe') and re.match(r'(R|F|S)\d', d['id']) and oid.get(d['id'])]
    print('start set', len(start))
    bw_today = set()
    try:
        for i in range(0, len(start), 40):
            bw_today |= ids_for('cited_by:' + '|'.join(start[i:i + 40]) + ',from_publication_date:2022-11-01')
    except RuntimeError: bw_today = earlier('backward_today_ids.json')
    old = {r['id'] for r in BW['records']}
    print(f'backward today {len(bw_today)}, 9 November {len(old)}, found both times {len(bw_today & old)}')
    json.dump(sorted(bw_today), open(LIVE / 'backward_today_ids.json', 'w'))

offline test run: this live internet step is skipped


## 7.3 Round 2, today

The start set is the round 1 snowballing papers included at full text.

In [5]:
if OFFLINE:
    print("offline test run: this live internet step is skipped")
else:
    ws = openpyxl.load_workbook(ROOT / 'tracker/master_tracker/Master_Tracker_LATEST.xlsx', read_only=True)['All_Records']
    it = ws.iter_rows(values_only=True); h = list(next(it)); ix = {k: i for i, k in enumerate(h)}
    r1 = [r[ix['OpenAlex ID']].replace('https://openalex.org/', '') for r in it if r[ix['Stage 2 decision']] == 'Include' and re.match(r'(F|S|B)\d', r[ix['ID']]) and r[ix['OpenAlex ID']]]
    print('round 2 start set', len(r1))
    r2_today = set()
    try:
        for i in range(0, len(r1), 40):
            g = '|'.join(r1[i:i + 40])
            r2_today |= ids_for('cited_by:' + g + ',from_publication_date:2022-11-01') | ids_for('cites:' + g + ',from_publication_date:2022-11-01')
    except RuntimeError: r2_today = earlier('round2_today_ids.json')
    old = {r['id'] for r in R2['records']}
    print(f'round 2 today {len(r2_today)}, 10 November {len(old)}, found both times {len(r2_today & old)}')
    json.dump(sorted(r2_today), open(LIVE / 'round2_today_ids.json', 'w'))

offline test run: this live internet step is skipped


## 7.3b Saved live runs

The counts of the latest saved live run of each direction (`snowballing/live_rerun_<date>/`), compared with the records of 9 and 10 November.

In [6]:
for name, fn, old_set, when in [('forward', 'forward_today_ids.json', {r['id'] for r in FW['records']} | {r['id'] for r in SA['records']}, '9 November'),
                                ('backward', 'backward_today_ids.json', {r['id'] for r in BW['records']}, '9 November'),
                                ('round 2', 'round2_today_ids.json', {r['id'] for r in R2['records']}, '10 November')]:
    c = sorted(glob.glob(str(ROOT / 'snowballing' / 'live_rerun_*' / fn)))
    if not c: print(f'{name}: no saved live run'); continue
    today = set(json.load(open(c[-1])))
    print(f'{name} (saved run {Path(c[-1]).parent.name}): today {len(today)}, {when} {len(old_set)}, found both times {len(today & old_set)}')

forward (saved run live_rerun_2026-11-15): today 4677, 9 November 4502, found both times 4495
backward (saved run live_rerun_2026-11-15): today 7523, 9 November 7419, found both times 7416
round 2 (saved run live_rerun_2026-11-15): today 9797, 10 November 9854, found both times 8651


## 7.4 Rounds 3 and later

Each later round starts from the papers newly included in the round before it and follows both their references and their
citing works, from 1 November 2022. The export of each round is kept in `snowballing/round_N/`. Script 26 makes the export
(or reuses the saved one) and finds the records not already held, by OpenAlex ID or by the same title; the new records are
numbered U0001 (round 3), V0001 (round 4) and W0001 (round 5). I screened every new record on title and
abstract with the AI tool under my eligibility rules, and a record kept at this stage went to the full text stage
(`r{N}_screen_batch_*_out.json`). Script 27 fetched the full texts of the kept records. The full text decisions of each round
were then made on the full text under the same rules (`s2ft_round{N}_*_out.json`).

### `scripts/26_snowball_round.py`

Snowballing round N: export from OpenAlex (or the saved export) and the records not already held, with IDs. Run below for each round.

In [7]:
%%writefile scripts/26_snowball_round.py
"""Snowballing round N (3 and later): backward (references) and forward (citing works) of the papers newly included in round N-1,
published from 2022-11-01, from OpenAlex. Saves the export and the new records (not in any earlier set) with IDs.

Usage: python 26_snowball_round.py <Report folder> <round> <prefix of the seed records> <prefix for new records>
Example: python 26_snowball_round.py Report 3 T U
Writes snowballing/round_N/openalex_snowball_roundN_<date>.json and snowballing/round_N/roundN_new_records_with_ids.json.
If the export already exists it is reused (never fetched twice)."""
import json, os, sys, re, glob, time, datetime, urllib.request, urllib.error
import openpyxl
R, N, SEEDP, NEWP = sys.argv[1], int(sys.argv[2]), sys.argv[3], sys.argv[4]
J = lambda p: json.load(open(p, encoding='utf-8'))
D = os.path.join(R, f'snowballing/round_{N}'); os.makedirs(D, exist_ok=True)
SEL = 'id,doi,title,publication_year,publication_date,type,cited_by_count,primary_location,best_oa_location,open_access,authorships,abstract_inverted_index'

def get(url, tries=6):
    for k in range(tries):
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'PRISMA-review-snowball (mailto:review@example.org)'})
            return json.load(urllib.request.urlopen(req, timeout=60))
        except urllib.error.HTTPError as e:
            if e.code == 429 and k == tries - 1: raise RuntimeError('OpenAlex daily limit')
            time.sleep(2 + 3 * k)
        except Exception:
            time.sleep(2 + 3 * k)
    raise RuntimeError('OpenAlex not reachable')

def abstract(inv):
    if not inv: return ''
    pos = sorted((p, w) for w, ps in inv.items() for p in ps)
    return ' '.join(w for _, w in pos)

def rec(w):
    loc = w.get('primary_location') or {}; src = (loc.get('source') or {})
    oa = w.get('open_access') or {}; best = w.get('best_oa_location') or {}
    return {'id': w['id'], 'doi': w.get('doi'), 'title': w.get('title'), 'year': w.get('publication_year'), 'date': w.get('publication_date'),
            'type': w.get('type'), 'cites': w.get('cited_by_count'), 'venue': src.get('display_name'), 'oa': bool(oa.get('is_oa')),
            'oa_url': best.get('pdf_url') or oa.get('oa_url') or best.get('landing_page_url'),
            'authors': '; '.join(a['author']['display_name'] for a in (w.get('authorships') or [])[:6] if a.get('author')),
            'abstract': abstract(w.get('abstract_inverted_index'))}

def fetch(filt):
    out, page = {}, 1
    while True:
        j = get(f'https://api.openalex.org/works?filter={filt}&per-page=200&select={SEL}&page={page}')
        for w in j['results']: out[w['id']] = w
        if page * 200 >= j['meta']['count'] or page >= 50: return out
        page += 1; time.sleep(0.15)

# seeds: records with the seed prefix included at full text in the current tracker
ws = openpyxl.load_workbook(os.path.join(R, 'tracker/master_tracker/Master_Tracker_LATEST.xlsx'), read_only=True)['All_Records']
it = ws.iter_rows(values_only=True); h = list(next(it)); ix = {k: i for i, k in enumerate(h)}
seeds = sorted({r[ix['OpenAlex ID']].replace('https://openalex.org/', '') for r in it
                if r[ix['Stage 2 decision']] == 'Include' and re.match(SEEDP + r'\d', r[ix['ID']]) and r[ix['OpenAlex ID']]})
print('seed papers', len(seeds))
ex = sorted(glob.glob(os.path.join(D, f'openalex_snowball_round{N}_*.json')))
if ex:
    E = J(ex[-1]); print('reusing', os.path.basename(ex[-1]))
    es = set(E.get('seed_ids', []))
    print(f'seed papers in the saved export: {len(es)} | in the tracker now: {len(seeds)} | in the export but no longer included: {len(es - set(seeds))} | included now but not in the export: {len(set(seeds) - es)}')
else:
    bw, fw = {}, {}
    for i in range(0, len(seeds), 40):
        g = '|'.join(seeds[i:i + 40])
        bw.update(fetch(f'cited_by:{g},from_publication_date:2022-11-01')); fw.update(fetch(f'cites:{g},from_publication_date:2022-11-01'))
        print('group', i // 40 + 1, 'backward', len(bw), 'forward', len(fw))
    now = datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
    recs = {}
    for d, s in (('backward', bw), ('forward', fw)):
        for k, w in s.items():
            r = recs.setdefault(k, rec(w)); r.setdefault('dirs', []).append(d)
    E = {'retrieved_utc': now, 'source': f'OpenAlex snowball round {N}: backward (cited_by) and forward (cites) of the {len(seeds)} papers newly included through round {N - 1}, from 2022-11-01',
         'log': [{'direction': 'backward', 'seed_papers': len(seeds), 'rows': len(bw), 'run_utc': now}, {'direction': 'forward', 'seed_papers': len(seeds), 'rows': len(fw), 'run_utc': now}],
         'seed_ids': seeds, 'records': list(recs.values())}
    json.dump(E, open(os.path.join(D, f'openalex_snowball_round{N}_{now[:10]}.json'), 'w', encoding='utf-8'), indent=1, ensure_ascii=False)
# records already held: every OpenAlex ID in the tracker, plus every earlier export
ws = openpyxl.load_workbook(os.path.join(R, 'tracker/master_tracker/Master_Tracker_LATEST.xlsx'), read_only=True)['All_Records']
it = ws.iter_rows(values_only=True); h = list(next(it)); ix = {k: i for i, k in enumerate(h)}
LATER = {3: 'U', 4: 'V', 5: 'W', 6: 'Y', 7: 'Z'}   # records of this round and later rounds are not 'already held' (so a rerun gives the same result)
own = tuple(p for n, p in LATER.items() if n >= N)
held = lambda i: not (own and re.match('(' + '|'.join(own) + r')\d', str(i)))
seen = {r[ix['OpenAlex ID']] for r in it if r[ix['OpenAlex ID']] and held(r[ix['ID']])}
titles = set()
ws = openpyxl.load_workbook(os.path.join(R, 'tracker/master_tracker/Master_Tracker_LATEST.xlsx'), read_only=True)['All_Records']
it = ws.iter_rows(values_only=True); next(it)
norm = lambda t: re.sub(r'[^a-z0-9]', '', (t or '').lower())
for r in it:
    if held(r[ix['ID']]): titles.add(norm(r[ix['Title']]))
for k in range(2, N):
    for f in glob.glob(os.path.join(R, f'snowballing/round_{k}/openalex_snowball_round{k}_*.json')):
        seen |= {r['id'] for r in J(f)['records']}
new, dup_title = [], 0
for r in E['records']:
    if r['id'] in seen: continue
    if norm(r['title']) and norm(r['title']) in titles: dup_title += 1; continue
    titles.add(norm(r['title'])); new.append(dict(r, **{f'r{N}id': f'{NEWP}{len(new) + 1:04d}'}))
json.dump(new, open(os.path.join(D, f'round{N}_new_records_with_ids.json'), 'w', encoding='utf-8'), indent=1, ensure_ascii=False)
print(f'round {N}: export {len(E["records"])} | already held {len(E["records"]) - len(new) - dup_title} | same title as a held record {dup_title} | new {len(new)}')

Overwriting scripts/26_snowball_round.py


### `scripts/27_round_full_texts.py`

Full texts of the records kept at title and abstract in a later round (arXiv copy, open access PDF, arXiv by title). It was run once per round; its log is `papers/retrieval_log/round<N>_full_text_log.json`, and the saved copies are reused here.

In [8]:
%%writefile scripts/27_round_full_texts.py
"""Full texts for the records kept at title and abstract in snowballing round N (3 and later).

Usage: python 27_round_full_texts.py <Report folder> <round>
For each kept record: (1) the arXiv copy named in its open access link, (2) its open access link when that is a PDF,
(3) the arXiv copy found by exact title (export.arxiv.org). A copy is kept only if it is a real PDF.
arXiv copies are saved as papers/pdfs/open_access/<ID>_arxiv_<arXiv id>.pdf (and in the project folder papers, so the
notebooks download them again from arXiv); other open access copies as <ID>_oa_<host>.pdf, also kept in
inputs/papers/pdfs_found_in_browser so that the full text rule (script 23) copies them in on every run.
Logged in papers/retrieval_log/round<N>_full_text_log.json."""
import os, re, sys, json, glob, time, shutil, urllib.request, urllib.parse
R, N = sys.argv[1], int(sys.argv[2])
J = lambda p: json.load(open(p, encoding='utf-8'))
PDF = os.path.join(R, 'papers/pdfs/open_access'); PROJ = os.path.join(os.path.dirname(os.path.abspath(R)), 'papers/pdfs/open_access')
MAN = os.path.join(R, 'inputs/papers/pdfs_found_in_browser')
recs = {r[f'r{N}id']: r for r in J(os.path.join(R, f'snowballing/round_{N}/round{N}_new_records_with_ids.json'))}
kept = [d['id'] for f in sorted(glob.glob(os.path.join(R, f'snowballing/round_{N}/r{N}_screen_batch_*_out.json'))) for d in J(f)['include_or_maybe']]
norm = lambda t: re.sub(r'[^a-z0-9]+', ' ', (t or '').lower()).strip()
def fetch(url):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0 (review reproduction)'})
    with urllib.request.urlopen(req, timeout=120) as r: return r.read()
def arxiv_by_title(title):
    q = urllib.parse.quote('ti:"' + re.sub(r'[^\w\s-]', ' ', title)[:200] + '"')
    xml = fetch(f'https://export.arxiv.org/api/query?search_query={q}&max_results=5').decode('utf-8', 'ignore')
    for e in re.findall(r'<entry>(.*?)</entry>', xml, re.S):
        t = re.search(r'<title>(.*?)</title>', e, re.S); i = re.search(r'<id>http://arxiv.org/abs/([^<]+)</id>', e)
        if t and i and norm(t.group(1)) == norm(title): return i.group(1).split('v')[0]
have = {f.split('_')[0] for f in os.listdir(PDF) if f.endswith('.pdf')}
log = {}
for i in kept:
    if i in have: log[i] = 'already saved'; continue
    r = recs[i]; url = r.get('oa_url') or ''; tries = []
    m = re.search(r'arxiv\.org/(?:abs|pdf)/(\d{4}\.\d{4,5})', url)
    if m: tries.append(('arxiv', m.group(1), f'https://export.arxiv.org/pdf/{m.group(1)}'))
    if url and not m: tries.append(('oa', re.sub(r'[^a-z0-9]+', '-', urllib.parse.urlparse(url).netloc.lower()).strip('-'), url))
    tries.append(('title', None, None))
    done = None
    for kind, tag, u in tries:
        try:
            if kind == 'title':
                ax = arxiv_by_title(r['title']); time.sleep(3)
                if not ax: continue
                kind, tag, u = 'arxiv', ax, f'https://export.arxiv.org/pdf/{ax}'
            data = fetch(u); time.sleep(3)
            if data[:4] != b'%PDF' or len(data) < 5000: continue
            fn = f'{i}_{kind}_{tag}.pdf'; open(os.path.join(PDF, fn), 'wb').write(data)
            if kind == 'arxiv' and os.path.isdir(PROJ): shutil.copy2(os.path.join(PDF, fn), os.path.join(PROJ, fn))
            if kind == 'oa': os.makedirs(MAN, exist_ok=True); shutil.copy2(os.path.join(PDF, fn), os.path.join(MAN, fn))
            done = f'{fn} from {u}'; break
        except Exception as e: log.setdefault(i + ' errors', []).append(f'{kind}: {type(e).__name__}')
    log[i] = done or 'no open access full text found'
    print(i, log[i][:90], flush=True)
json.dump(log, open(os.path.join(R, f'papers/retrieval_log/round{N}_full_text_log.json'), 'w'), indent=1)
print('kept', len(kept), '| with full text', sum(1 for i in kept if not log[i].startswith('no ')))

Overwriting scripts/27_round_full_texts.py


In [9]:
LATERP = {3: ('T', 'U'), 4: ('U', 'V'), 5: ('V', 'W'), 6: ('W', 'Y'), 7: ('Y', 'Z')}
for n, (sp, npre) in LATERP.items():
    if not (ROOT / f'snowballing/round_{n}').exists() or not glob.glob(str(ROOT / f'snowballing/round_{n}/openalex_snowball_round{n}_*.json')): continue
    saved = J(f'snowballing/round_{n}/round{n}_new_records_with_ids.json')
    p = run('26_snowball_round.py', str(n), sp, npre)
    again = J(f'snowballing/round_{n}/round{n}_new_records_with_ids.json')
    check(f'round {n}: the new records found again are the saved ones', [r['id'] for r in again] == [r['id'] for r in saved], len(again))
    ag = J(f'snowballing/round_{n}/r{n}_stage1_agreement.json')
    print(f'round {n} stage 1: two screeners agreed on {ag["agreement"] * 100:.1f} percent of {ag["records"]} records (kappa {ag["kappa"]:.2f})')

--- 26_snowball_round.py: exit code 0, 9.1 s ---
seed papers 123
reusing openalex_snowball_round3_2026-11-15.json
seed papers in the saved export: 121 | in the tracker now: 123 | in the export but no longer included: 2 | included now but not in the export: 4
round 3: export 1877 | already held 1228 | same title as a held record 5 | new 644
PASS  26_snowball_round.py ran without error
PASS  round 3: the new records found again are the saved ones   [644]
round 3 stage 1: two screeners agreed on 98.8 percent of 644 records (kappa 0.91)


--- 26_snowball_round.py: exit code 0, 8.9 s ---
seed papers 23
reusing openalex_snowball_round4_2026-11-15.json
seed papers in the saved export: 27 | in the tracker now: 23 | in the export but no longer included: 4 | included now but not in the export: 0
round 4: export 512 | already held 333 | same title as a held record 1 | new 178
PASS  26_snowball_round.py ran without error
PASS  round 4: the new records found again are the saved ones   [178]
round 4 stage 1: two screeners agreed on 98.3 percent of 178 records (kappa 0.89)


--- 26_snowball_round.py: exit code 0, 8.9 s ---
seed papers 8
reusing openalex_snowball_round5_2026-11-15.json
seed papers in the saved export: 10 | in the tracker now: 8 | in the export but no longer included: 2 | included now but not in the export: 0
round 5: export 140 | already held 88 | same title as a held record 1 | new 51
PASS  26_snowball_round.py ran without error
PASS  round 5: the new records found again are the saved ones   [51]
round 5 stage 1: two screeners agreed on 98.0 percent of 51 records (kappa 0.85)


## 7.5 Yield per round (from the final decisions)

In [10]:
import openpyxl
ws = openpyxl.load_workbook(ROOT / 'tracker/master_tracker/Master_Tracker_LATEST.xlsx', read_only=True)['All_Records']
it = ws.iter_rows(values_only=True); h = list(next(it)); ix = {k: i for i, k in enumerate(h)}
grp = {'R': 'database search', 'GS': 'database search', 'F': 'round 1', 'S': 'round 1', 'B': 'round 1', 'T': 'round 2', 'U': 'round 3', 'V': 'round 4', 'W': 'round 5', 'Y': 'round 6', 'Z': 'round 7'}
scr, keep, inc = collections.Counter(), collections.Counter(), collections.Counter()
for r in it:
    g = grp.get(re.match(r'[A-Z]+', r[ix['ID']]).group(0))
    if not g or r[ix['Stage 1 decision']] in (None, 'Duplicate (removed before screening)'): continue
    scr[g] += 1; keep[g] += r[ix['Stage 1 decision']] in ('Include', 'Maybe'); inc[g] += r[ix['Stage 2 decision']] == 'Include'
for g in [g for g in ['database search', 'round 1', 'round 2', 'round 3', 'round 4', 'round 5', 'round 6', 'round 7'] if scr[g]]:
    print(f'{g:16} screened {scr[g]:6,} | kept at stage 1 {keep[g]:5,} ({100 * keep[g] / scr[g]:.1f} percent) | included {inc[g]:5,}')
expect('round 1 yield, percent kept at stage 1', round(100 * keep['round 1'] / scr['round 1'], 1), 7.1)
expect('round 2 yield, percent kept at stage 1', round(100 * keep['round 2'] / scr['round 2'], 1), 4.5)
last = [g for g in ['round 2', 'round 3', 'round 4', 'round 5', 'round 6', 'round 7'] if scr[g]][-1]
print(f'{last} kept {keep[last]} records at stage 1 and included {inc[last]} at full text: ' + ('snowballing is saturated (a round with no new included paper).' if inc[last] == 0 else 'snowballing is not fully saturated.'))

database search  screened  3,316 | kept at stage 1   860 (25.9 percent) | included   495
round 1          screened  9,727 | kept at stage 1   690 (7.1 percent) | included   356
round 2          screened  4,717 | kept at stage 1   210 (4.5 percent) | included   123
round 3          screened    644 | kept at stage 1    50 (7.8 percent) | included    23
round 4          screened    178 | kept at stage 1    16 (9.0 percent) | included     8
round 5          screened     51 | kept at stage 1     4 (7.8 percent) | included     0
PASS  round 1 yield, percent kept at stage 1: 7.1   [7.1]
PASS  round 2 yield, percent kept at stage 1: 4.5   [4.5]
round 5 kept 4 records at stage 1 and included 0 at full text: snowballing is saturated (a round with no new included paper).


In [11]:
save_checks('07_Snowballing')

8 of 8 checks pass in 07_Snowballing
